# Õppetund 02 - Microsoft Agendi raamistikuga tutvumine

**Microsoft Agendi raamistiku (MAF)** on ühtne raamistik tehisintellekti agentide loomiseks. See pakub selget, komposiitset arhitektuuri nelja põhikomponendiga:

- **Klient** – ühendub tehisintellekti mudeli lõpp-punktiga ja haldab suhtlust
- **Agent** – kapseldab kliendi juhiste ja tööriistade definitsioonidega
- **Tööriistad** – laiendavad agendi võimeid kohandatud funktsioonidega, mida mudel saab kutsuda
- **Sessioon** – säilitab vestluse ajaloo mitme vooruga interaktsioonide jaoks

Selles õppetükis ehitame **reisi broneerimise agendi**, mis kontrollib sihtkoha saadavust nende kontseptide abil.


## Seadistamine


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agentraamistiku arhitektuuri mõistmine

Microsoft Agent Framework järgib kihilist arhitektuuri:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `AzureAIProjectAgentProvider` ühendub Azure OpenAI juurutusega. See haldab autentimist, päringute vormindamist ja vastuste analüüsimist.
2. **Agent** – Luuakse kliendilt `provider.create_agent()` kaudu, agent ühendab mudeli juurdepääsu juhistega (süsteemi prompt) ja tööriistadega.
3. **Tööriistad** – Python funktsioonid, mis on tähistatud `@tool` ja mida agent saab kutsuda tegevuste sooritamiseks või andmete hankimiseks.
4. **Seanss** – `AgentSession` objekt (loodud `agent.create_session()` kaudu), mis salvestab vestluse ajaloo, võimaldades mitme vooruga dialoogi, kus agent mäletab eelnevat konteksti.

Lähme iga kihi ehitamise juurde samm-sammult.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tööriistade lisamine @tool dekoratiiviga

Tööriistad võimaldavad agentidel teostada tegevusi, mis ületavad teksti genereerimist. `@tool` dekoratiiv teisendab tavalise Pythoni funktsiooni millekski, mida agent saab kutsuda.

Olulised punktid:
- Kasutage `Annotated[type, "kirjeldus"]`, et mudel mõistaks iga parameetrit.
- Docstring muutub tööriista kirjelduseks, mida mudel näeb.
- `approval_mode="never_require"` tähendab, et tööriist käivitatakse automaatselt ilma kasutaja kinnitusteta.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agendi loomine tööriistadega

Nüüd ühendame kliendi, juhised ja tööriistad agendiks. `instructions` toimivad süsteemi käsuna — need määravad agendi isikuomadused ja käitumise.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mitme vooruga vestlused sessioonidega

`AgentSession` (loodud `agent.create_session()` abil) hoiab vestluse kõikide sõnumite kohta arvestust. Sama sessiooni edastades iga `agent.run()` kutsumise jaoks, on agendil juurdepääs kogu vestluse ajaloole ja ta saab viidata varasematele sõnumitele.

Edastame `tools=[check_destination_availability]`, et agent saaks iga vooru ajal kutsuda meie saadavuse kontrollijat.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Kokkuvõte

Selles õppetükis uurisite Microsoft Agent Framework nelja tugisammast:

| Mõiste | Mida õppisite |
|---------|------------------|
| **Kliendi** | `AzureAIProjectAgentProvider` ühendub Azure OpenAI-ga mandaadipõhise autentimisega |
| **Agent** | `provider.create_agent()` ühendab mudeli ühenduse juhiste ja nimega |
| **Tööriistad** | `@tool` dekoratiiv võimaldab agendil kutsuda Pythoni funktsioone |
| **Seanss** | `agent.create_session()` säilitab vestluse ajaloo mitme vooru jooksul |

Need põhikomponendid kokku moodustavad agendi, kes suudab pidada loomulikku vestlust, kutsuda väliseid funktsioone ja säilitada konteksti — alus keerukamate agendimustrite jaoks tulevastes õppetükkides.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud AI tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüame tagada täpsust, olge teadlikud, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise info puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste või valearusaamade eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
